# Phase 6: Model Application - LightGBM Timestomping Detection

This notebook applies the trained LightGBM model to analyze feature-extracted files and produce detection outputs suitable for Autopsy plugin integration.

## Inputs
- Trained LightGBM model (`model_lightgbm.joblib`)
- Feature scaler (`feature_scaler.joblib`)
- Feature CSV from Phase 3 (`file_features_*.csv`)

## Outputs
1. `detected_files.csv` - Files flagged as timestomped with detection explanations
2. `files_with_features.csv` - All analyzed files with confidence scores
3. `summary.txt` - Human-readable detection report


In [41]:
# Cell 2: Import Libraries
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully")


Libraries imported successfully


In [42]:
# Cell 3: Configuration

# Model paths
MODEL_DIR = Path("/Users/soni/Github/Digital-Detectives_Thesis/data/Phase 4: Model Training/final version")
MODEL_PATH = MODEL_DIR / "model_lightgbm.joblib"
SCALER_PATH = MODEL_DIR / "feature_scaler.joblib"

# Input data path
INPUT_CSV = Path("/Users/soni/Github/Digital-Detectives_Thesis/data/Phase 3: Feature Engineering & Labeling/v1/file_features_LoneWolf.csv")

# Output directory
OUTPUT_DIR = Path("/Users/soni/Github/Digital-Detectives_Thesis/data/Phase 6: Output Sample")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Model configuration
THRESHOLD = 0.02  # LightGBM optimal threshold from training
DATASET_NAME = INPUT_CSV.stem.replace("file_features_", "")

print(f"Model:      {MODEL_PATH}")
print(f"Scaler:     {SCALER_PATH}")
print(f"Input:      {INPUT_CSV}")
print(f"Output:     {OUTPUT_DIR}")
print(f"Threshold:  {THRESHOLD}")
print(f"Dataset:    {DATASET_NAME}")


Model:      /Users/soni/Github/Digital-Detectives_Thesis/data/Phase 4: Model Training/final version/model_lightgbm.joblib
Scaler:     /Users/soni/Github/Digital-Detectives_Thesis/data/Phase 4: Model Training/final version/feature_scaler.joblib
Input:      /Users/soni/Github/Digital-Detectives_Thesis/data/Phase 3: Feature Engineering & Labeling/v1/file_features_LoneWolf.csv
Output:     /Users/soni/Github/Digital-Detectives_Thesis/data/Phase 6: Output Sample
Threshold:  0.02
Dataset:    LoneWolf


## 1. Load Model and Scaler


In [43]:
# Cell 5: Load trained model and scaler
model = joblib.load(MODEL_PATH)
scaler = joblib.load(SCALER_PATH)

print(f"Model loaded: {type(model).__name__}")
print(f"Scaler loaded: {type(scaler).__name__}")
print(f"Model expects {model.n_features_in_} features")


Model loaded: LGBMClassifier
Scaler loaded: StandardScaler
Model expects 29 features


## 2. Load and Prepare Input Data


In [44]:
# Cell 7: Load input CSV
df_input = pd.read_csv(INPUT_CSV)

print(f"Input data loaded: {len(df_input):,} files")
print(f"Columns: {len(df_input.columns)}")
print(f"\nFirst few columns: {list(df_input.columns[:10])}")


Input data loaded: 17,897 files
Columns: 33

First few columns: ['FileFRN', 'dataID', 'FileName', 'FilePath', 'num_timestamp_changes', 'num_backward_jumps', 'num_forward_jumps', 'num_creation_changes', 'max_backward_jump_seconds', 'mean_jump_seconds']


In [45]:
# Cell 8: Define feature columns (must match training order)
FEATURE_COLUMNS = [
    'num_timestamp_changes',
    'num_backward_jumps',
    'num_forward_jumps',
    'num_creation_changes',
    'max_backward_jump_seconds',
    'mean_jump_seconds',
    'timestamp_change_density',
    'num_zero_nanosecond_events',
    'only_SI_modified',
    'num_update_resident_value',
    'repeated_update_resident_value',
    'consecutive_timestamp_changes',
    'num_logfile_events',
    'has_logfile_ts_change',
    'has_usn_basic_info',
    'has_usn_close',
    'has_usn_file_create',
    'num_usn_basic_info',
    'num_usn_close',
    'num_usn_file_create',
    'logfile_usn_mismatch',
    'has_usn_basic_pattern',
    'num_usnjrnl_events',
    'min_inter_event_delta',
    'max_inter_event_delta',
    'mean_inter_event_delta',
    'burstiness_score',
    'event_time_span_seconds',
    'total_events'
]

# Columns to exclude from features (metadata)
EXCLUDE_COLUMNS = ['dataID', 'FileName', 'is_timestomped', 'FilePath']

print(f"Feature columns defined: {len(FEATURE_COLUMNS)}")


Feature columns defined: 29


In [46]:
# Cell 9: Prepare feature matrix

# Check for missing columns
missing_cols = [col for col in FEATURE_COLUMNS if col not in df_input.columns]
if missing_cols:
    print(f"WARNING: Missing columns: {missing_cols}")
else:
    print("All required feature columns present")

# Extract features
X = df_input[FEATURE_COLUMNS].copy()

# Handle missing values
X = X.fillna(0)

# Convert boolean columns to int
bool_cols = X.select_dtypes(include=['bool']).columns
X[bool_cols] = X[bool_cols].astype(int)

print(f"\nFeature matrix shape: {X.shape}")
print(f"Data types: {X.dtypes.value_counts().to_dict()}")


All required feature columns present

Feature matrix shape: (17897, 29)
Data types: {dtype('int64'): 15, dtype('float64'): 14}


## 3. Run Model Predictions


In [47]:
# Cell 11: Generate predictions

# Get probability scores (confidence)
y_prob = model.predict_proba(X)[:, 1]

# Apply threshold to get binary predictions
y_pred = (y_prob >= THRESHOLD).astype(int)

# Add results to dataframe
df_results = df_input.copy()
df_results['Confidence'] = y_prob
df_results['Flagged'] = y_pred.astype(bool)

print(f"Predictions complete")
print(f"Total files:    {len(df_results):,}")
print(f"Files flagged:  {df_results['Flagged'].sum():,} ({df_results['Flagged'].mean()*100:.2f}%)")
print(f"Files cleared:  {(~df_results['Flagged']).sum():,}")


Predictions complete
Total files:    17,897
Files flagged:  331 (1.85%)
Files cleared:  17,566


In [48]:
# Cell 12: Analyze confidence distribution

print("Confidence Distribution:")
print("=" * 50)

# High confidence (>0.5)
high_conf = (df_results['Confidence'] > 0.5).sum()
print(f"High Confidence (>0.5):      {high_conf:,}")

# Medium confidence (0.1 - 0.5)
med_conf = ((df_results['Confidence'] > 0.1) & (df_results['Confidence'] <= 0.5)).sum()
print(f"Medium Confidence (0.1-0.5): {med_conf:,}")

# Low confidence (threshold - 0.1)
low_conf = ((df_results['Confidence'] >= THRESHOLD) & (df_results['Confidence'] <= 0.1)).sum()
print(f"Low Confidence ({THRESHOLD}-0.1):   {low_conf:,}")

# Below threshold
below_thresh = (df_results['Confidence'] < THRESHOLD).sum()
print(f"Below Threshold (<{THRESHOLD}):   {below_thresh:,}")


Confidence Distribution:
High Confidence (>0.5):      320
Medium Confidence (0.1-0.5): 1
Low Confidence (0.02-0.1):   10
Below Threshold (<0.02):   17,566


## 4. Generate Detection Reasons


In [49]:
# Cell 14: Define enhanced detection reason generator

def convert_seconds_to_readable(seconds):
    """Convert seconds to human-readable duration."""
    if seconds >= 86400:
        return f"{seconds/86400:.1f} days"
    elif seconds >= 3600:
        return f"{seconds/3600:.1f} hours"
    elif seconds >= 60:
        return f"{seconds/60:.1f} minutes"
    else:
        return f"{seconds:.0f} seconds"

def generate_detection_reasons_enhanced(row):
    """Generate forensically-contextualized detection reasons."""
    indicators = []
    
    # Backward timestamp jumps
    if row['num_backward_jumps'] > 0:
        max_jump_sec = row['max_backward_jump_seconds']
        duration = convert_seconds_to_readable(max_jump_sec)
        severity = 'HIGH' if max_jump_sec > 86400 else 'MEDIUM'
        
        indicators.append({
            'type': 'BACKWARD_TIMESTAMP',
            'severity': severity,
            'finding': f"{int(row['num_backward_jumps'])} backward timestamp jump(s) detected (max: {duration})",
            'forensic_meaning': "Timestamps moved backwards in time, which cannot occur through normal file "
                               "operations. This indicates deliberate manipulation to make a file appear older "
                               "than it actually is. Attackers use this technique to blend malicious files with "
                               "legitimate system files by matching their timestamps.",
            'artifact_source': '$MFT $STANDARD_INFORMATION timestamps'
        })
    
    # Zero nanosecond timestamps
    if row['num_zero_nanosecond_events'] > 0:
        indicators.append({
            'type': 'ZERO_NANOSECONDS',
            'severity': 'MEDIUM',
            'finding': f"{int(row['num_zero_nanosecond_events'])} timestamp event(s) with zero nanosecond precision",
            'forensic_meaning': "NTFS stores timestamps with 100-nanosecond precision. Legitimate file operations "
                               "produce non-zero nanosecond values due to system clock granularity. Zero nanoseconds "
                               "typically indicates timestamps were set programmatically using APIs like SetFileTime() "
                               "with rounded values, or timestomping tools (Timestomp, SetMACE, NewFileTime) that "
                               "fail to populate sub-second precision.",
            'artifact_source': '$MFT timestamp fields (nanosecond component)'
        })
    
    # SI-only modification
    if row['only_SI_modified'] == 1 or row['only_SI_modified'] == True:
        indicators.append({
            'type': 'SI_ONLY_MODIFICATION',
            'severity': 'HIGH',
            'finding': "Only $STANDARD_INFORMATION timestamps modified, $FILE_NAME unchanged",
            'forensic_meaning': "Normal file operations update both $STANDARD_INFORMATION (SI) and $FILE_NAME (FN) "
                               "attributes. Most timestomping tools only modify SI because FN requires kernel-level "
                               "access. When SI shows different timestamps than FN, the FN attribute reveals the "
                               "true file creation time. This is a strong indicator of timestamp manipulation.",
            'artifact_source': '$MFT $STANDARD_INFORMATION vs $FILE_NAME comparison'
        })
    
    # LogFile/USN mismatch
    if row['logfile_usn_mismatch'] == 1 or row['logfile_usn_mismatch'] == True:
        indicators.append({
            'type': 'ARTIFACT_MISMATCH',
            'severity': 'HIGH',
            'finding': "$LogFile and $UsnJrnl show inconsistent timestamp records",
            'forensic_meaning': "The $LogFile (transaction journal) and $UsnJrnl (change journal) independently "
                               "record file system operations with their own timestamps. When these artifacts show "
                               "different timestamps than the $MFT for the same file operation, it indicates the "
                               "$MFT timestamps were modified AFTER the original operation was journaled. Attackers "
                               "rarely modify journal entries as it requires advanced techniques and risks corruption.",
            'artifact_source': '$LogFile and $UsnJrnl cross-correlation with $MFT'
        })
    
    # High burstiness
    if row['burstiness_score'] > 0.5:
        severity = 'HIGH' if row['burstiness_score'] > 0.8 else 'MEDIUM'
        indicators.append({
            'type': 'HIGH_BURSTINESS',
            'severity': severity,
            'finding': f"Temporal clustering score: {row['burstiness_score']:.3f}",
            'forensic_meaning': "Multiple timestamp modifications occurred in rapid succession (burst pattern). "
                               "Normal file operations produce timestamp changes distributed over time as users "
                               "interact with files naturally. High burstiness suggests automated or scripted "
                               "timestamp manipulation where multiple timestamps were modified in a short time "
                               "window, typical of batch timestomping operations.",
            'artifact_source': '$MFT timestamp change event timing analysis'
        })
    
    # Consecutive timestamp changes
    if row['consecutive_timestamp_changes'] > 2:
        indicators.append({
            'type': 'CONSECUTIVE_CHANGES',
            'severity': 'MEDIUM',
            'finding': f"{int(row['consecutive_timestamp_changes'])} consecutive timestamp modifications without content changes",
            'forensic_meaning': "Multiple sequential modifications to timestamp fields were detected without "
                               "intervening file content changes. Normal file access patterns show content "
                               "modifications (reads, writes) between timestamp updates. Consecutive timestamp-only "
                               "changes indicate deliberate timestamp manipulation rather than normal file usage.",
            'artifact_source': '$MFT and $LogFile event sequence analysis'
        })
    
    # High number of timestamp changes
    if row['num_timestamp_changes'] > 10:
        indicators.append({
            'type': 'EXCESSIVE_CHANGES',
            'severity': 'LOW',
            'finding': f"{int(row['num_timestamp_changes'])} total timestamp change events recorded",
            'forensic_meaning': "An unusually high number of timestamp modifications were recorded for this file. "
                               "While not definitive on its own, excessive timestamp changes combined with other "
                               "indicators may suggest repeated manipulation attempts or automated tools cycling "
                               "through timestamp values.",
            'artifact_source': '$MFT and $LogFile event count'
        })
    
    return indicators

def format_short_reason(indicators):
    """Format indicators into short CSV-friendly string."""
    if not indicators:
        return ""
    parts = []
    for ind in indicators:
        parts.append(f"{ind['type']}: {ind['finding']}")
    return "; ".join(parts)

def get_overall_severity(indicators, confidence):
    """Determine overall severity from indicators AND confidence score.
    
    Severity Logic:
    - High confidence (>0.5): Trust indicator severity fully
    - Medium confidence (0.1-0.5): Downgrade severity by one level
    - Low confidence (<0.1): Cap at LOW regardless of indicators
    
    This prevents low-confidence detections from being labeled HIGH severity.
    """
    if not indicators:
        return "LOW"
    
    # Get maximum indicator severity
    indicator_severities = [ind['severity'] for ind in indicators]
    if 'HIGH' in indicator_severities:
        max_indicator_severity = 'HIGH'
    elif 'MEDIUM' in indicator_severities:
        max_indicator_severity = 'MEDIUM'
    else:
        max_indicator_severity = 'LOW'
    
    # Modulate severity based on confidence
    if confidence > 0.5:
        # High confidence: trust indicator severity fully
        return max_indicator_severity
    elif confidence > 0.1:
        # Medium confidence: downgrade by one level
        if max_indicator_severity == 'HIGH':
            return 'MEDIUM'
        elif max_indicator_severity == 'MEDIUM':
            return 'LOW'
        else:
            return 'LOW'
    else:
        # Low confidence (<0.1): cap at LOW regardless of indicators
        return 'LOW'

def get_recommended_action(indicators, severity):
    """Generate recommended action based on indicators and severity."""
    if not indicators:
        return "No action required"
    
    types = [ind['type'] for ind in indicators]
    
    if severity == 'HIGH':
        if 'SI_ONLY_MODIFICATION' in types:
            return "VERIFY: Compare $STANDARD_INFORMATION and $FILE_NAME timestamps using MFT parser"
        elif 'ARTIFACT_MISMATCH' in types:
            return "VERIFY: Cross-reference $LogFile and $UsnJrnl entries for this file"
        elif 'BACKWARD_TIMESTAMP' in types:
            return "INVESTIGATE: Examine file metadata and correlate with timeline analysis"
        else:
            return "INVESTIGATE: High confidence detection requires manual verification"
    elif severity == 'MEDIUM':
        return "REVIEW: Manual inspection recommended to confirm timestomping"
    else:
        return "MONITOR: Low confidence detection, consider in context of other findings"

# Step 1: Apply enhanced detection to create Indicators column FIRST
df_results['Indicators'] = df_results.apply(generate_detection_reasons_enhanced, axis=1)

# Step 2: Create Detection_Reasons from Indicators
df_results['Detection_Reasons'] = df_results['Indicators'].apply(format_short_reason)

# Step 3: Calculate Indicator_Count
df_results['Indicator_Count'] = df_results['Indicators'].apply(len)

# Step 4: Calculate Severity using BOTH indicators AND confidence
df_results['Severity'] = df_results.apply(
    lambda row: get_overall_severity(row['Indicators'], row['Confidence']), 
    axis=1
)

# Step 5: Generate Recommended_Action based on indicators and severity
df_results['Recommended_Action'] = df_results.apply(
    lambda row: get_recommended_action(row['Indicators'], row['Severity']),
    axis=1
)

print("Enhanced detection reasons generated")
print(f"Files with indicators: {(df_results['Indicator_Count'] > 0).sum():,}")
print(f"\nSeverity Distribution (Flagged Files):")
print(df_results[df_results['Flagged']]['Severity'].value_counts())


Enhanced detection reasons generated
Files with indicators: 14,615

Severity Distribution (Flagged Files):
Severity
HIGH      320
LOW        10
MEDIUM      1
Name: count, dtype: int64


## 5. Calculate Forensic Metrics


In [50]:
# Cell 16: Calculate forensic metrics

total_files = len(df_results)
flagged_files = df_results['Flagged'].sum()
cleared_files = total_files - flagged_files

# Candidate Reduction Rate (CRR)
crr = cleared_files / total_files

# Estimated NNI (assuming ~12 true positives based on LoneWolf ground truth)
# In production, this would be unknown
estimated_true_positives = 12  # LoneWolf has 12 known timestomped files
nni = flagged_files / estimated_true_positives if estimated_true_positives > 0 else 0

# Flag rate
flag_rate = flagged_files / total_files

print("Forensic Metrics:")
print("=" * 50)
print(f"Total Files Analyzed:        {total_files:,}")
print(f"Files Flagged:               {flagged_files:,} ({flag_rate*100:.2f}%)")
print(f"Files Cleared:               {cleared_files:,}")
print(f"Candidate Reduction Rate:    {crr*100:.2f}%")
print(f"Estimated NNI:               {nni:.1f} files/detection")


Forensic Metrics:
Total Files Analyzed:        17,897
Files Flagged:               331 (1.85%)
Files Cleared:               17,566
Candidate Reduction Rate:    98.15%
Estimated NNI:               27.6 files/detection


## 6. Generate Output Files


In [51]:
# Cell 18: Output 1 - detected_files.csv (Enhanced)

# Filter flagged files only
df_detected = df_results[df_results['Flagged']].copy()

# Create forensic summary for each file
def create_forensic_summary(row):
    """Create a brief forensic summary statement."""
    indicators = row['Indicators']
    if not indicators:
        return "Pattern match suggests possible timestomping"
    
    summaries = []
    for ind in indicators:
        if ind['type'] == 'BACKWARD_TIMESTAMP':
            summaries.append(f"timestamps manipulated backwards ({ind['finding'].split('max: ')[1].rstrip(')')})")
        elif ind['type'] == 'ZERO_NANOSECONDS':
            summaries.append("zero nanosecond precision detected")
        elif ind['type'] == 'SI_ONLY_MODIFICATION':
            summaries.append("$SI modified but $FN unchanged")
        elif ind['type'] == 'ARTIFACT_MISMATCH':
            summaries.append("journal artifacts inconsistent")
        elif ind['type'] == 'HIGH_BURSTINESS':
            summaries.append("burst pattern in modifications")
    
    return "File shows: " + "; ".join(summaries) if summaries else "Multiple indicators suggest timestomping"

df_detected['Forensic_Summary'] = df_detected.apply(create_forensic_summary, axis=1)

# Calculate backward jump duration in readable format
df_detected['Backward_Jump_Duration'] = df_detected['max_backward_jump_seconds'].apply(
    lambda x: convert_seconds_to_readable(x) if x > 0 else "N/A"
)

# Select and order columns for output
detected_columns = [
    'FileName',
    'Confidence',
    'Severity',
    'Indicator_Count',
    'Forensic_Summary',
    'Detection_Reasons',
    'Recommended_Action',
    'num_backward_jumps',
    'Backward_Jump_Duration',
    'num_zero_nanosecond_events',
    'only_SI_modified',
    'logfile_usn_mismatch',
    'burstiness_score',
    'consecutive_timestamp_changes'
]

# Add FilePath if available
if 'FilePath' in df_detected.columns:
    detected_columns = ['FilePath'] + detected_columns

df_detected_output = df_detected[detected_columns].sort_values(
    ['Severity', 'Confidence'], 
    ascending=[True, False],  # HIGH severity first, then by confidence
    key=lambda x: x.map({'HIGH': 0, 'MEDIUM': 1, 'LOW': 2}) if x.name == 'Severity' else x
)

# Re-sort properly
severity_order = {'HIGH': 0, 'MEDIUM': 1, 'LOW': 2}
df_detected_output['_sort'] = df_detected_output['Severity'].map(severity_order)
df_detected_output = df_detected_output.sort_values(['_sort', 'Confidence'], ascending=[True, False])
df_detected_output = df_detected_output.drop('_sort', axis=1)

# Save to CSV
detected_path = OUTPUT_DIR / f"detected_files_{DATASET_NAME}.csv"
df_detected_output.to_csv(detected_path, index=False)

print(f"Output 1: detected_files_{DATASET_NAME}.csv")
print(f"  Location: {detected_path}")
print(f"  Records:  {len(df_detected_output):,}")
print(f"\nSeverity Breakdown:")
print(df_detected_output['Severity'].value_counts())
print(f"\nTop 5 Detected Files:")
print(df_detected_output.head()[['FileName', 'Confidence', 'Severity', 'Forensic_Summary']])


Output 1: detected_files_LoneWolf.csv
  Location: /Users/soni/Github/Digital-Detectives_Thesis/data/Phase 6: Output Sample/detected_files_LoneWolf.csv
  Records:  331

Severity Breakdown:
Severity
HIGH      320
LOW        10
MEDIUM      1
Name: count, dtype: int64

Top 5 Detected Files:
               FileName  Confidence Severity  \
13357     DeathToll.jpg    0.999999     HIGH   
13349      DemLogic.jpg    0.999999     HIGH   
13450  BladeofGrass.jpg    0.999999     HIGH   
13340      DarkWolf.png    0.999999     HIGH   
13365     Planning.docx    0.999997     HIGH   

                                        Forensic_Summary  
13357  File shows: timestamps manipulated backwards (...  
13349  File shows: timestamps manipulated backwards (...  
13450  File shows: timestamps manipulated backwards (...  
13340  File shows: timestamps manipulated backwards (...  
13365  File shows: timestamps manipulated backwards (...  


In [52]:
# Cell 19: Output 2 - files_with_features.csv

# Select columns for full output
full_output_columns = ['FileName', 'Confidence', 'Flagged'] + FEATURE_COLUMNS

# Add FilePath if available
if 'FilePath' in df_results.columns:
    full_output_columns = ['FilePath'] + full_output_columns

# Add ground truth if available (for validation)
if 'is_timestomped' in df_results.columns:
    full_output_columns.append('is_timestomped')

df_full_output = df_results[full_output_columns].sort_values('Confidence', ascending=False)

# Save to CSV
full_path = OUTPUT_DIR / f"files_with_features_{DATASET_NAME}.csv"
df_full_output.to_csv(full_path, index=False)

print(f"Output 2: files_with_features_{DATASET_NAME}.csv")
print(f"  Location: {full_path}")
print(f"  Records:  {len(df_full_output):,}")
print(f"  Columns:  {len(full_output_columns)}")


Output 2: files_with_features_LoneWolf.csv
  Location: /Users/soni/Github/Digital-Detectives_Thesis/data/Phase 6: Output Sample/files_with_features_LoneWolf.csv
  Records:  17,897
  Columns:  33


## 6.2 Output 2b - Detection Details Report (Detailed Forensic Explanations)


In [53]:
# Cell 19.6: Output 2b - detection_details.txt (Full forensic explanations)

def generate_detailed_report(df_flagged):
    """Generate detailed forensic report for each flagged file."""
    
    report = f"""================================================================================
NTFS TIMESTOMPING DETECTION - DETAILED FINDINGS
================================================================================
Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
Dataset:   {DATASET_NAME}
Model:     LightGBM (Phase 4 Baseline)
Threshold: {THRESHOLD}
================================================================================

Total Files Flagged: {len(df_flagged):,}
  - HIGH Severity:   {(df_flagged['Severity'] == 'HIGH').sum()}
  - MEDIUM Severity: {(df_flagged['Severity'] == 'MEDIUM').sum()}
  - LOW Severity:    {(df_flagged['Severity'] == 'LOW').sum()}

================================================================================
DETAILED ANALYSIS PER FILE
================================================================================
"""
    
    # Sort by severity and confidence
    severity_order = {'HIGH': 0, 'MEDIUM': 1, 'LOW': 2}
    df_sorted = df_flagged.copy()
    df_sorted['_sort'] = df_sorted['Severity'].map(severity_order)
    df_sorted = df_sorted.sort_values(['_sort', 'Confidence'], ascending=[True, False])
    
    for idx, (_, row) in enumerate(df_sorted.iterrows(), 1):
        report += f"""
{'='*80}
[{idx}/{len(df_sorted)}] {row['FileName']}
{'='*80}
Confidence Score: {row['Confidence']:.6f}
Overall Severity: {row['Severity']}
Indicators Found: {row['Indicator_Count']}
"""
        if 'FilePath' in row and pd.notna(row.get('FilePath')):
            report += f"File Path:        {row['FilePath']}\n"
        
        indicators = row['Indicators']
        
        if indicators:
            for i, ind in enumerate(indicators, 1):
                report += f"""
INDICATOR {i}: {ind['type']} [{ind['severity']} SEVERITY]
{'-'*80}
Finding:
  {ind['finding']}

Forensic Significance:
  {ind['forensic_meaning']}

Artifact Source:
  {ind['artifact_source']}
"""
        else:
            report += """
INDICATOR: PATTERN_MATCH [LOW SEVERITY]
{'-'*80}
Finding:
  Combination of features matches known timestomping signatures

Forensic Significance:
  While no single indicator exceeded detection thresholds, the overall
  feature pattern is consistent with timestomped files in the training
  dataset. This may indicate a sophisticated or partial timestomping
  attempt that avoided obvious indicators.

Artifact Source:
  Machine learning pattern matching across 29 features
"""
        
        report += f"""
RECOMMENDED ACTION:
{'-'*80}
  {row['Recommended_Action']}

RAW FEATURE VALUES:
{'-'*80}
  Backward Jumps:           {int(row['num_backward_jumps'])}
  Max Backward Jump:        {convert_seconds_to_readable(row['max_backward_jump_seconds']) if row['max_backward_jump_seconds'] > 0 else 'N/A'}
  Zero Nanosecond Events:   {int(row['num_zero_nanosecond_events'])}
  SI-Only Modified:         {row['only_SI_modified']}
  LogFile/USN Mismatch:     {row['logfile_usn_mismatch']}
  Burstiness Score:         {row['burstiness_score']:.4f}
  Consecutive Changes:      {int(row['consecutive_timestamp_changes'])}
  Total Timestamp Changes:  {int(row['num_timestamp_changes'])}
"""
    
    report += f"""
{'='*80}
END OF DETAILED FINDINGS
{'='*80}

METHODOLOGY NOTES
{'='*80}
This analysis uses a LightGBM gradient boosting model trained on 22 datasets
containing 52 verified timestomped files from real-world APT campaigns and
controlled experiments. The model analyzes 29 features derived from:

  - $MFT (Master File Table): Timestamp values and change patterns
  - $LogFile: Transaction journal entries
  - $UsnJrnl: Change journal records
  - Cross-artifact correlation: Consistency checks between artifacts

Key detection principles:
  1. Timestamp Physics: Backward jumps violate temporal causality
  2. Precision Artifacts: Sub-second precision reveals tool signatures  
  3. Attribute Divergence: SI/FN discrepancies indicate selective modification
  4. Journal Immutability: Journal entries preserve pre-manipulation state

False Positive Considerations:
  - System restore operations may trigger backward timestamp indicators
  - File copy operations preserve source timestamps (legitimate)
  - Virtual machine snapshots may cause timestamp anomalies
  - Time zone changes can appear as timestamp modifications

Always correlate findings with case context and timeline analysis.

================================================================================
Generated by Digital Detectives Timestomping Detector v1.0
================================================================================
"""
    
    return report

# Generate detailed report
df_flagged = df_results[df_results['Flagged']].copy()
detailed_report = generate_detailed_report(df_flagged)

# Save detailed report
details_path = OUTPUT_DIR / f"detection_details_{DATASET_NAME}.txt"
with open(details_path, 'w') as f:
    f.write(detailed_report)

print(f"Output 2b: detection_details_{DATASET_NAME}.txt")
print(f"  Location: {details_path}")
print(f"\nPreview (first 3000 characters):")
print(detailed_report[:3000])


Output 2b: detection_details_LoneWolf.txt
  Location: /Users/soni/Github/Digital-Detectives_Thesis/data/Phase 6: Output Sample/detection_details_LoneWolf.txt

Preview (first 3000 characters):
NTFS TIMESTOMPING DETECTION - DETAILED FINDINGS
Generated: 2026-01-13 08:44:37
Dataset:   LoneWolf
Model:     LightGBM (Phase 4 Baseline)
Threshold: 0.02

Total Files Flagged: 331
  - HIGH Severity:   320
  - MEDIUM Severity: 1
  - LOW Severity:    10

DETAILED ANALYSIS PER FILE

[1/331] DeathToll.jpg
Confidence Score: 0.999999
Overall Severity: HIGH
Indicators Found: 4
File Path:        /Users/jcloudy/Dropbox/DeathToll.jpg

INDICATOR 1: BACKWARD_TIMESTAMP [HIGH SEVERITY]
--------------------------------------------------------------------------------
Finding:
  1 backward timestamp jump(s) detected (max: 1.4 days)

Forensic Significance:
  Timestamps moved backwards in time, which cannot occur through normal file operations. This indicates deliberate manipulation to make a file appear older than 

In [54]:
# Cell 20: Output 3 - summary.txt (Enhanced)

# Get top 10 suspicious files
top_10 = df_results.nlargest(10, 'Confidence')[['FileName', 'Confidence', 'Severity', 'Indicator_Count']]

# Count detection patterns
backward_jumps = (df_results['num_backward_jumps'] > 0).sum()
zero_nanoseconds = (df_results['num_zero_nanosecond_events'] > 0).sum()
si_only = (df_results['only_SI_modified'] == 1).sum() if 'only_SI_modified' in df_results.columns else 0
logfile_mismatch = (df_results['logfile_usn_mismatch'] == 1).sum() if 'logfile_usn_mismatch' in df_results.columns else 0
high_burstiness = (df_results['burstiness_score'] > 0.5).sum()

# Confidence and severity counts
high_conf = (df_results['Confidence'] > 0.5).sum()
med_conf = ((df_results['Confidence'] > 0.1) & (df_results['Confidence'] <= 0.5)).sum()
low_conf = ((df_results['Confidence'] >= THRESHOLD) & (df_results['Confidence'] <= 0.1)).sum()

high_sev = (df_results[df_results['Flagged']]['Severity'] == 'HIGH').sum()
med_sev = (df_results[df_results['Flagged']]['Severity'] == 'MEDIUM').sum()
low_sev = (df_results[df_results['Flagged']]['Severity'] == 'LOW').sum()

# Generate summary report
summary_report = f"""================================================================================
NTFS TIMESTOMPING DETECTION REPORT
================================================================================

Analysis Date:      {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
Dataset Analyzed:   {DATASET_NAME}
Model Used:         LightGBM (Phase 4 Baseline)
Threshold:          {THRESHOLD}

--------------------------------------------------------------------------------
EXECUTIVE SUMMARY
--------------------------------------------------------------------------------
Total Files Analyzed:       {total_files:,}
Files Flagged:              {flagged_files:,} ({flag_rate*100:.2f}%)

Confidence Breakdown:
  High Confidence (>0.5):     {high_conf}
  Medium Confidence (0.1-0.5): {med_conf}
  Low Confidence ({THRESHOLD}-0.1):  {low_conf}

Severity Breakdown:
  HIGH Severity:   {high_sev} (requires immediate investigation)
  MEDIUM Severity: {med_sev} (manual review recommended)
  LOW Severity:    {low_sev} (consider in context)

Candidate Reduction Rate:   {crr*100:.2f}%
  (Analyst reviews {flagged_files:,} files instead of {total_files:,})

Number Needed to Investigate: ~{nni:.1f} files per expected true positive

--------------------------------------------------------------------------------
TOP 10 MOST SUSPICIOUS FILES
--------------------------------------------------------------------------------
Rank  Confidence  Severity  Indicators  FileName
----  ----------  --------  ----------  --------
"""

for i, (_, row) in enumerate(top_10.iterrows(), 1):
    summary_report += f"{i:<5} {row['Confidence']:.6f}  {row['Severity']:<8}  {int(row['Indicator_Count']):<10}  {row['FileName']}\n"

summary_report += f"""
--------------------------------------------------------------------------------
DETECTION INDICATOR SUMMARY
--------------------------------------------------------------------------------
The following indicators were detected across all analyzed files:

Indicator Type                    Count    Description
------------------------------    -----    -----------
Backward Timestamp Jumps          {backward_jumps:<5}    Files where timestamps moved backwards
Zero Nanosecond Precision         {zero_nanoseconds:<5}    Files with suspiciously round timestamps
SI-Only Modifications             {si_only:<5}    $STANDARD_INFORMATION modified, $FILE_NAME unchanged
LogFile/USN Mismatch              {logfile_mismatch:<5}    Journal artifacts inconsistent with $MFT
High Burstiness (>0.5)            {high_burstiness:<5}    Rapid successive timestamp modifications

--------------------------------------------------------------------------------
INDICATOR EXPLANATIONS
--------------------------------------------------------------------------------

BACKWARD_TIMESTAMP (HIGH Severity when >1 day)
  Timestamps that move backwards in time cannot occur through normal file
  operations. This definitively indicates deliberate manipulation to make
  files appear older than they actually are.

ZERO_NANOSECONDS (MEDIUM Severity)
  NTFS stores 100-nanosecond precision. Legitimate operations produce non-zero
  values. Zero nanoseconds indicates programmatic timestamp setting, typically
  via timestomping tools that don't populate sub-second precision.

SI_ONLY_MODIFICATION (HIGH Severity)
  Normal operations update both $STANDARD_INFORMATION and $FILE_NAME attributes.
  When only $SI is modified, it indicates tools that bypass the file system.
  The $FILE_NAME attribute retains the original timestamp.

ARTIFACT_MISMATCH (HIGH Severity)
  $LogFile and $UsnJrnl independently record operations. Inconsistency with
  $MFT timestamps indicates post-operation modification. Journal entries are
  difficult to modify without detection.

HIGH_BURSTINESS (MEDIUM-HIGH Severity)
  Multiple timestamp changes in rapid succession suggest automated or scripted
  manipulation. Normal file usage produces distributed timestamp changes.

--------------------------------------------------------------------------------
RECOMMENDED NEXT STEPS
--------------------------------------------------------------------------------
1. Review HIGH severity files immediately - these show strong manipulation signs
2. For BACKWARD_TIMESTAMP findings, compare $SI and $FN timestamps manually
3. Cross-reference flagged files with case timeline
4. Consider context: system restore, VM snapshots may cause false positives
5. Export detailed findings from detection_details_{DATASET_NAME}.txt

--------------------------------------------------------------------------------
METHODOLOGY
--------------------------------------------------------------------------------
This analysis uses machine learning (LightGBM) trained on 22 datasets with
52 known timestomped files from real APT campaigns. The model achieves 100%
recall on validation datasets, ensuring no known timestomped files are missed.

Features analyzed: {len(FEATURE_COLUMNS)} (timestamp patterns, artifact consistency, temporal behavior)
Training data: APT17, APT19, APT21, APT28, APT29, APT30, APT37, APT38, APT40,
               DarkHotel, Kimsuky, Winnti, and controlled timestomping experiments

Based on: Oh et al. (2024) NTFS timestomping detection methodology

--------------------------------------------------------------------------------
OUTPUT FILES GENERATED
--------------------------------------------------------------------------------
1. detected_files_{DATASET_NAME}.csv      - {flagged_files:,} flagged files with explanations
2. files_with_features_{DATASET_NAME}.csv - {total_files:,} files with all features  
3. detection_details_{DATASET_NAME}.txt   - Detailed forensic analysis per file
4. summary_{DATASET_NAME}.txt             - This report

================================================================================
Generated by Digital Detectives Timestomping Detector v1.0
================================================================================
"""

# Save summary
summary_path = OUTPUT_DIR / f"summary_{DATASET_NAME}.txt"
with open(summary_path, 'w') as f:
    f.write(summary_report)

print(f"Output 3: summary_{DATASET_NAME}.txt")
print(f"  Location: {summary_path}")
print("\n" + "=" * 80)
print(summary_report)


Output 3: summary_LoneWolf.txt
  Location: /Users/soni/Github/Digital-Detectives_Thesis/data/Phase 6: Output Sample/summary_LoneWolf.txt

NTFS TIMESTOMPING DETECTION REPORT

Analysis Date:      2026-01-13 08:44:37
Dataset Analyzed:   LoneWolf
Model Used:         LightGBM (Phase 4 Baseline)
Threshold:          0.02

--------------------------------------------------------------------------------
EXECUTIVE SUMMARY
--------------------------------------------------------------------------------
Total Files Analyzed:       17,897
Files Flagged:              331 (1.85%)

Confidence Breakdown:
  High Confidence (>0.5):     320
  Medium Confidence (0.1-0.5): 1
  Low Confidence (0.02-0.1):  10

Severity Breakdown:
  HIGH Severity:   320 (requires immediate investigation)
  MEDIUM Severity: 1 (manual review recommended)
  LOW Severity:    10 (consider in context)

Candidate Reduction Rate:   98.15%
  (Analyst reviews 331 files instead of 17,897)

Number Needed to Investigate: ~27.6 files per ex

## 7. Validation Against Ground Truth (Optional)


In [55]:
# Cell 22: Validate against ground truth (if available)

if 'is_timestomped' in df_results.columns:
    print("Ground Truth Validation:")
    print("=" * 50)
    
    # Ground truth labels
    y_true = df_results['is_timestomped'].fillna(0).astype(int)
    y_pred = df_results['Flagged'].astype(int)
    y_prob = df_results['Confidence']
    
    # Calculate metrics
    TP = ((y_true == 1) & (y_pred == 1)).sum()
    FN = ((y_true == 1) & (y_pred == 0)).sum()
    FP = ((y_true == 0) & (y_pred == 1)).sum()
    TN = ((y_true == 0) & (y_pred == 0)).sum()
    
    total_positive = y_true.sum()
    
    # Forensic metrics
    recall = TP / total_positive if total_positive > 0 else 0
    crr_actual = TN / (TN + FP) if (TN + FP) > 0 else 0
    nni_actual = (TP + FP) / TP if TP > 0 else float('inf')
    fpr = FP / (FP + TN) if (FP + TN) > 0 else 0
    
    print(f"\nConfusion Matrix:")
    print(f"  TP (Detected):       {TP}")
    print(f"  FN (Missed):         {FN}")
    print(f"  FP (False Alarms):   {FP:,}")
    print(f"  TN (Correct Normal): {TN:,}")
    
    print(f"\nForensic Metrics:")
    print(f"  Recall:              {recall:.4f} ({TP}/{total_positive} detected)")
    print(f"  CRR:                 {crr_actual:.4f} ({crr_actual*100:.2f}% excluded)")
    print(f"  NNI:                 {nni_actual:.2f} files/detection")
    print(f"  FPR:                 {fpr:.6f}")
    
    if FN > 0:
        print(f"\nWARNING: {FN} timestomped file(s) missed!")
        missed = df_results[(y_true == 1) & (y_pred == 0)][['FileName', 'Confidence']]
        print("Missed files:")
        print(missed)
    else:
        print(f"\nSUCCESS: All {total_positive} timestomped files detected!")
    
    # Show detected timestomped files
    print(f"\nDetected Timestomped Files:")
    detected_gt = df_results[(y_true == 1) & (y_pred == 1)][['FileName', 'Confidence', 'Detection_Reasons']]
    print(detected_gt.sort_values('Confidence', ascending=False))
    
else:
    print("No ground truth column ('is_timestomped') found in input data.")
    print("Skipping validation step.")


No ground truth column ('is_timestomped') found in input data.
Skipping validation step.


## 8. Summary and Output Locations


In [56]:
# Cell 24: Final summary

print("=" * 80)
print("PHASE 6 MODEL APPLICATION COMPLETE")
print("=" * 80)

print(f"\nDataset: {DATASET_NAME}")
print(f"Total Files Analyzed: {total_files:,}")
print(f"Files Flagged: {flagged_files:,} ({flag_rate*100:.2f}%)")
print(f"Candidate Reduction Rate: {crr*100:.2f}%")

print(f"\nSeverity Distribution:")
print(f"  HIGH:   {high_sev}")
print(f"  MEDIUM: {med_sev}")
print(f"  LOW:    {low_sev}")

print(f"\nOutput Files Generated:")
print(f"  1. {OUTPUT_DIR / f'detected_files_{DATASET_NAME}.csv'}")
print(f"  2. {OUTPUT_DIR / f'files_with_features_{DATASET_NAME}.csv'}")
print(f"  3. {OUTPUT_DIR / f'detection_details_{DATASET_NAME}.txt'}")
print(f"  4. {OUTPUT_DIR / f'summary_{DATASET_NAME}.txt'}")

print("\nThese outputs provide:")
print("  - detected_files.csv:      Quick triage list with severity ratings")
print("  - files_with_features.csv: Full data for further analysis")
print("  - detection_details.txt:   Forensic explanations for each flagged file")
print("  - summary.txt:             Executive summary and methodology")

print("\n" + "=" * 80)


PHASE 6 MODEL APPLICATION COMPLETE

Dataset: LoneWolf
Total Files Analyzed: 17,897
Files Flagged: 331 (1.85%)
Candidate Reduction Rate: 98.15%

Severity Distribution:
  HIGH:   320
  MEDIUM: 1
  LOW:    10

Output Files Generated:
  1. /Users/soni/Github/Digital-Detectives_Thesis/data/Phase 6: Output Sample/detected_files_LoneWolf.csv
  2. /Users/soni/Github/Digital-Detectives_Thesis/data/Phase 6: Output Sample/files_with_features_LoneWolf.csv
  3. /Users/soni/Github/Digital-Detectives_Thesis/data/Phase 6: Output Sample/detection_details_LoneWolf.txt
  4. /Users/soni/Github/Digital-Detectives_Thesis/data/Phase 6: Output Sample/summary_LoneWolf.txt

These outputs provide:
  - detected_files.csv:      Quick triage list with severity ratings
  - files_with_features.csv: Full data for further analysis
  - detection_details.txt:   Forensic explanations for each flagged file
  - summary.txt:             Executive summary and methodology

